In [3]:
import arcpy
import os

def generate_non_coral_random_points():
    # ================= 參數設定區 =================
    boundary_fc = "OL_P1_plot_6m"      # 1. 限制範圍圖層
    coral_t1 = "OL_P1_2204_coral_Simplify"           # 2. T1 珊瑚圖層
    coral_t2 = "OL_P1_2209_cor_SimplifyPolyg"           # 3. T2 珊瑚圖層
    num_points = 200                         # 4. 隨機點數量
    min_dist = "0.1 Meters"                  # 5. 選擇性：點與點之間最小距離 (例如 10公分)
    
    output_folder = arcpy.env.workspace      # 輸出位置 (預設為當前 GDB)
    out_points_name = "Random_Points_Empty"  # 輸出點圖層名稱
    # =============================================

    arcpy.env.overwriteOutput = True
    
    try:
        print("步驟 1/3: 正在合併 T1 與 T2 珊瑚範圍...")
        combined_coral = "memory/combined_coral"
        arcpy.analysis.Union([coral_t1, coral_t2], combined_coral)
        
        dissolved_coral = "memory/dissolved_coral"
        arcpy.management.Dissolve(combined_coral, dissolved_coral)

        print("步驟 2/3: 正在從邊界中扣除珊瑚範圍以取得空地...")
        # 
        empty_area = "memory/empty_area"
        arcpy.analysis.Erase(boundary_fc, dissolved_coral, empty_area)

        print(f"步驟 3/3: 正在產生 {num_points} 個隨機點...")
        # 修正關鍵：將參數名稱改為 constraining_feature_class
        arcpy.management.CreateRandomPoints(
            out_path=output_folder, 
            out_name=out_points_name, 
            constraining_feature_class=empty_area, # 這裡修正了！
            number_of_points_or_field=num_points,
            minimum_allowed_distance=min_dist      # 加入最小距離確保採樣均勻
        )

        print(f"✅ 成功！隨機點已儲存於: {os.path.join(str(output_folder), out_points_name)}")

    except arcpy.ExecuteError:
        print(arcpy.GetMessages(2))
    except Exception as e:
        print(f"發生錯誤: {e}")

if __name__ == "__main__":
    generate_non_coral_random_points()

步驟 1/3: 正在合併 T1 與 T2 珊瑚範圍...
步驟 2/3: 正在從邊界中扣除珊瑚範圍以取得空地...
步驟 3/3: 正在產生 200 個隨機點...
✅ 成功！隨機點已儲存於: D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\Random_Points_Empty


In [12]:
import arcpy
import os

def main():
    # ================= 參數設定區 =================
    # 輸入改為剛產生的隨機點圖層
    input_points = "Random_Points_Boundary"  
    out_square_fc = "Random_Squares_Boundary_2204"
    
    # 正方形邊長列表 (單位: 公分)
    square_sizes_cm = [2, 4, 8, 16, 32, 64, 128]
    
    # 設定環境
    arcpy.env.overwriteOutput = True
    desc = arcpy.Describe(input_points)
    sr = desc.spatialReference
    
    # 單位換算 (公分轉地圖單位)
    unit_factor = 0.01 if sr.linearUnitName.lower() in ['meter', 'metre'] else 1.0
    if "foot" in sr.linearUnitName.lower():
        unit_factor = 0.0328084

    # ================= 1. 建立輸出圖層 =================
    print(f"正在建立方格圖層: {out_square_fc}...")
    arcpy.management.CreateFeatureclass(arcpy.env.workspace, out_square_fc, "POLYGON", spatial_reference=sr)
    
    # 簡化後的欄位：原始點ID、邊長等級
    arcpy.management.AddField(out_square_fc, "Parent_Point_ID", "LONG")
    arcpy.management.AddField(out_square_fc, "Size_Level_cm", "SHORT")
    arcpy.management.AddField(out_square_fc, "Orig_Area", "DOUBLE") # 設為0，保持結構一致

    # ================= 2. 讀取點並生成方格 =================
    # 只需要讀取點的 ID 和 座標
    search_fields = ["OID@", "SHAPE@XY"]
    insert_fields = ["SHAPE@", "Parent_Point_ID", "Size_Level_cm", "Orig_Area"]

    print("正在處理點位並生成正方形序列...")
    
    
    with arcpy.da.InsertCursor(out_square_fc, insert_fields) as i_cursor:
        with arcpy.da.SearchCursor(input_points, search_fields) as s_cursor:
            for row in s_cursor:
                point_id, (cx, cy) = row
                
                for size_cm in square_sizes_cm:
                    # 計算半邊長
                    half_side = (size_cm * unit_factor) / 2.0
                    
                    # 建立四個頂點
                    p1 = arcpy.Point(cx - half_side, cy - half_side)
                    p2 = arcpy.Point(cx - half_side, cy + half_side)
                    p3 = arcpy.Point(cx + half_side, cy + half_side)
                    p4 = arcpy.Point(cx + half_side, cy - half_side)
                    
                    # 建立多邊形幾何
                    square_geom = arcpy.Polygon(arcpy.Array([p1, p2, p3, p4, p1]), sr)
                    
                    # 寫入資料 (Orig_Area 填 0)
                    i_cursor.insertRow([square_geom, point_id, size_cm, 0.0])

    print(f"🎉 成功！已針對點圖層生成 {len(square_sizes_cm)} 個不同尺寸的方格序列。")

if __name__ == '__main__':
    main()

正在建立方格圖層: Random_Squares_Boundary_2204...
正在處理點位並生成正方形序列...
🎉 成功！已針對點圖層生成 7 個不同尺寸的方格序列。


In [5]:
import arcpy
import os

# --- 1. 設定參數 ---
# 確保 rect_fc 是你剛產生的隨機點方格圖層名稱
rect_fc = "Random_Squares_Sequence_2204" 
overlap_fc_t1 = "OL_P1_2204_coral_Simplify"  # 第 1 個時段 (T1)
overlap_fc_t2 = "OL_P1_2209_cor_SimplifyPolyg"  # 第 2 個時段 (T2)
boundary_fc = "OL_P1_plot_6m"
form_field = "Form"

# 輸出 CSV 設定
output_folder = r"D:\ArcGIS\Morpho_Spatial_Demography\output" 
csv_name = "Random_Squares_Cover_Analysis.csv"

# ID 與欄位參數 (關鍵修改點)
id_part1 = "Parent_Point_ID" # 改為隨機點產生的 ID 欄位
id_part2 = "Size_Level_cm"
custom_uid_field = "Recruit_UID" 
orig_area_field = "Orig_Area" # 在隨機方格中此值應為 0

# 新增計算欄位名稱
t2_orig_gap_field = "T2_Orig_Gap"
net_growth_field = "Net_Growth"

area_unit = "SQUARE_METERS"
out_boundary_field = "Exceeds_Boundary"

arcpy.env.overwriteOutput = True

# --- 2. 建立合併的唯一 ID (Recruit_UID) ---
if not arcpy.ListFields(rect_fc, custom_uid_field):
    arcpy.management.AddField(rect_fc, custom_uid_field, "TEXT", field_length=100)

print(f"正在根據 {id_part1} 更新 Recruit_UID...")
expression = f"str(!{id_part1}!) + '_' + str(!{id_part2}!)"
arcpy.management.CalculateField(rect_fc, custom_uid_field, expression, "PYTHON3")

# --- 3. 自動讀取 Domain ---
def get_domain_map(fc, field_name):
    desc = arcpy.Describe(fc)
    d_name = next((f.domain for f in desc.fields if f.name == field_name), "")
    if d_name:
        workspace = desc.path
        domains = arcpy.da.ListDomains(workspace)
        for d in domains:
            if d.name == d_name:
                return {k: arcpy.ValidateFieldName(v) for k, v in d.codedValues.items()}
    return {}

domain_map = get_domain_map(overlap_fc_t1, form_field)
all_forms = list(domain_map.values())
print(f"解析到 {len(all_forms)} 種珊瑚類別。")

# --- 4. 定義運算函數 ---
def run_intersection(in_fc, suffix):
    print(f"正在運算 {suffix} 的重疊面積...")
    temp_table = f"memory/table_{suffix}"
    arcpy.analysis.TabulateIntersection(
        in_zone_features=rect_fc,
        zone_fields=custom_uid_field,
        in_class_features=in_fc,
        out_table=temp_table,
        class_fields=form_field,
        out_units=area_unit
    )
    
    period_results = {}
    total_col = f"Total_Area_{suffix}"
    
    with arcpy.da.SearchCursor(temp_table, [custom_uid_field, form_field, "AREA"]) as sc:
        for row in sc:
            uid, f_code, area = row
            f_alias = f"{domain_map.get(f_code, str(f_code))}_{suffix}"
            if uid not in period_results:
                period_results[uid] = {total_col: 0}
            period_results[uid][f_alias] = area
            period_results[uid][total_col] += area
    return period_results

# 執行 T1 與 T2 運算
results_t1 = run_intersection(overlap_fc_t1, "T1")
results_t2 = run_intersection(overlap_fc_t2, "T2")

# --- 5. 準備所有欄位 ---
t1_fields = [f"{f}_T1" for f in all_forms] + ["Total_Area_T1"]
t2_fields = [f"{f}_T2" for f in all_forms] + ["Total_Area_T2"]
calc_fields = [t2_orig_gap_field, net_growth_field]

for f in t1_fields + t2_fields + calc_fields:
    if not arcpy.ListFields(rect_fc, f):
        arcpy.management.AddField(rect_fc, f, "DOUBLE")

if not arcpy.ListFields(rect_fc, out_boundary_field):
    arcpy.management.AddField(rect_fc, out_boundary_field, "SHORT")

# --- 6. 判斷範圍 ---
print("檢查範圍邊界...")
rect_lyr = arcpy.management.MakeFeatureLayer(rect_fc, "rect_lyr")
arcpy.management.SelectLayerByLocation(rect_lyr, "COMPLETELY_WITHIN", boundary_fc)
within_uids = {row[0] for row in arcpy.da.SearchCursor(rect_lyr, [custom_uid_field])}

# --- 7. 更新數據與數學計算 ---
print("正在計算並寫入數據...")
update_fields = [custom_uid_field, out_boundary_field, orig_area_field] + t1_fields + t2_fields + calc_fields
f_idx = {name: i for i, name in enumerate(update_fields)}

with arcpy.da.UpdateCursor(rect_fc, update_fields) as cursor:
    for row in cursor:
        uid = row[f_idx[custom_uid_field]]
        # 取得原始面積，隨機點方格通常為 0
        orig_area = row[f_idx[orig_area_field]] if row[f_idx[orig_area_field]] else 0
        
        # A. 範圍標記
        row[f_idx[out_boundary_field]] = 0 if uid in within_uids else 1
        
        # B. 寫入 T1 數據
        area_t1 = 0
        if uid in results_t1:
            area_t1 = results_t1[uid].get("Total_Area_T1", 0)
            for f in t1_fields:
                row[f_idx[f]] = results_t1[uid].get(f, 0)
        else:
            for f in t1_fields: row[f_idx[f]] = 0
            
        # C. 寫入 T2 數據
        area_t2 = 0
        if uid in results_t2:
            area_t2 = results_t2[uid].get("Total_Area_T2", 0)
            for f in t2_fields:
                row[f_idx[f]] = results_t2[uid].get(f, 0)
        else:
            for f in t2_fields: row[f_idx[f]] = 0
        
        # D. 數學計算 (針對空地的生長分析)
        # 1. T2 的覆蓋情況
        row[f_idx[t2_orig_gap_field]] = area_t2 - orig_area
        
        # 2. 淨變化量 (T2 - T1 - 初始)
        row[f_idx[net_growth_field]] = area_t2 - area_t1 - orig_area
            
        cursor.updateRow(row)

# --- 8. 輸出 CSV ---
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

csv_output_full = os.path.join(output_folder, csv_name)
print(f"正在匯出隨機方格分析 CSV 到: {csv_output_full}")

arcpy.conversion.TableToTable(rect_fc, output_folder, csv_name)

arcpy.management.Delete("rect_lyr")
print("🎉 隨機方格 T1/T2 監測任務完成！")

正在根據 Parent_Point_ID 更新 Recruit_UID...
解析到 12 種珊瑚類別。
正在運算 T1 的重疊面積...
正在運算 T2 的重疊面積...
檢查範圍邊界...
正在計算並寫入數據...
正在匯出隨機方格分析 CSV 到: D:\ArcGIS\Morpho_Spatial_Demography\output\Random_Squares_Cover_Analysis.csv
🎉 隨機方格 T1/T2 監測任務完成！


In [3]:
import arcpy
import os
from arcpy.sa import *

# --- 0. 環境與地圖設定 ---
try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    active_map = aprx.activeMap
    default_gdb = aprx.defaultGeodatabase
    arcpy.env.workspace = default_gdb
    arcpy.env.overwriteOutput = True
    
    print("="*40)
    print(f"🌟 啟用的地圖: {active_map.name if active_map else 'None'}")
    print(f"📂 當前資料庫 (GDB): {default_gdb}")
    print("="*40)
except Exception as e:
    print(f"⚠️ 無法連接至 ArcGIS Project (可能不在 Pro 內執行): {e}")

# --- 1. 設定參數 ---
rect_fc = "Random_Squares_Boundary_2204" 
overlap_fc_t1 = "OL_P1_2204_coral_Simplify" 
overlap_fc_t2 = "OL_P1_2209_cor_SimplifyPolyg" 
boundary_fc = "OL_P1_plot_6m"
form_field = "Form"

# --- 新增：Raster 分析參數 ---
run_raster_analysis = True  # 設定為 False 可跳過此分析
raster_input = "OL_P1_2204_Crevice_NC" 
crevice_amount_field = "crevice_amount"
crevice_area_field = "Crevice_area"
crevice_percent_field = "crevice_percent"

# 輸出 CSV 設定
output_folder = r"D:\ArcGIS\Morpho_Spatial_Demography\output" 
csv_name = "Random_Squares_Cover_Analysis.csv"

# ID 與欄位參數
id_part1 = "Parent_Point_ID" 
id_part2 = "Size_Level_cm"
custom_uid_field = "Recruit_UID" 
orig_area_field = "Orig_Area" 

# 新增計算欄位名稱
t2_orig_gap_field = "T2_Orig_Gap"
net_growth_field = "Net_Growth"

area_unit = "SQUARE_METERS"
out_boundary_field = "Exceeds_Boundary"

# --- 2. 建立合併的唯一 ID (Recruit_UID) ---
if not arcpy.ListFields(rect_fc, custom_uid_field):
    arcpy.management.AddField(rect_fc, custom_uid_field, "TEXT", field_length=100)

print(f"正在根據 {id_part1} 更新 Recruit_UID...")
expression = f"str(!{id_part1}!) + '_' + str(!{id_part2}!)"
arcpy.management.CalculateField(rect_fc, custom_uid_field, expression, "PYTHON3")

# --- 3. 自動讀取 Domain ---
def get_domain_map(fc, field_name):
    desc = arcpy.Describe(fc)
    d_name = next((f.domain for f in desc.fields if f.name == field_name), "")
    if d_name:
        workspace = desc.path
        domains = arcpy.da.ListDomains(workspace)
        for d in domains:
            if d.name == d_name:
                return {k: arcpy.ValidateFieldName(v) for k, v in d.codedValues.items()}
    return {}

domain_map = get_domain_map(overlap_fc_t1, form_field)
all_forms = list(domain_map.values())
print(f"解析到 {len(all_forms)} 種珊瑚類別。")

# --- 4. 定義運算函數 ---
def run_intersection(in_fc, suffix):
    print(f"正在運算 {suffix} 的重疊面積...")
    temp_table = f"memory/table_{suffix}"
    arcpy.analysis.TabulateIntersection(
        in_zone_features=rect_fc,
        zone_fields=custom_uid_field,
        in_class_features=in_fc,
        out_table=temp_table,
        class_fields=form_field,
        out_units=area_unit
    )
    
    period_results = {}
    total_col = f"Total_Area_{suffix}"
    
    with arcpy.da.SearchCursor(temp_table, [custom_uid_field, form_field, "AREA"]) as sc:
        for row in sc:
            uid, f_code, area = row
            f_alias = f"{domain_map.get(f_code, str(f_code))}_{suffix}"
            if uid not in period_results:
                period_results[uid] = {total_col: 0}
            period_results[uid][f_alias] = area
            period_results[uid][total_col] += area
    return period_results

# 執行 T1 與 T2 運算
results_t1 = run_intersection(overlap_fc_t1, "T1")
results_t2 = run_intersection(overlap_fc_t2, "T2")

# --- 5. Raster Crevice 分析 (新的邏輯套用) ---
crevice_results = {}
pixel_area = 0
if run_raster_analysis:
    print("正在執行 Raster Crevice 分析...")
    if arcpy.CheckExtension("Spatial") == "Available":
        arcpy.CheckOutExtension("Spatial")
        
        # 取得解析度並計算單像素面積
        desc_r = arcpy.Describe(raster_input)
        pw, ph = desc_r.meanCellWidth, desc_r.meanCellHeight
        pixel_area = pw * ph
        print(f"📊 Raster 解析度: {pw:.4f} x {ph:.4f} m | 單像素面積: {pixel_area:.6f} m²")
        
        zonal_table = "memory/zonal_crevice"
        ZonalStatisticsAsTable(rect_fc, custom_uid_field, raster_input, zonal_table, "DATA", "SUM")
        
        with arcpy.da.SearchCursor(zonal_table, [custom_uid_field, "SUM"]) as sc:
            for uid, pixel_sum in sc:
                crevice_results[uid] = pixel_sum if pixel_sum else 0
        arcpy.CheckInExtension("Spatial")
    else:
        print("⚠️ 缺少 Spatial Analyst 授權，跳過 Raster 分析。")
        run_raster_analysis = False

# --- 6. 準備所有欄位 ---
t1_fields = [f"{f}_T1" for f in all_forms] + ["Total_Area_T1"]
t2_fields = [f"{f}_T2" for f in all_forms] + ["Total_Area_T2"]
calc_fields = [t2_orig_gap_field, net_growth_field]

if run_raster_analysis:
    calc_fields += [crevice_amount_field, crevice_area_field, crevice_percent_field]

for f in t1_fields + t2_fields + calc_fields:
    if not arcpy.ListFields(rect_fc, f):
        arcpy.management.AddField(rect_fc, f, "DOUBLE")

if not arcpy.ListFields(rect_fc, out_boundary_field):
    arcpy.management.AddField(rect_fc, out_boundary_field, "SHORT")

# --- 7. 判斷範圍 ---
print("檢查範圍邊界...")
rect_lyr = arcpy.management.MakeFeatureLayer(rect_fc, "rect_lyr")
arcpy.management.SelectLayerByLocation(rect_lyr, "COMPLETELY_WITHIN", boundary_fc)
within_uids = {row[0] for row in arcpy.da.SearchCursor(rect_lyr, [custom_uid_field])}

# --- 8. 更新數據與數學計算 ---
print("正在計算並寫入數據...")
# 包含 SHAPE@AREA 用於百分比計算分母
update_fields = [custom_uid_field, out_boundary_field, orig_area_field, "SHAPE@AREA"] + t1_fields + t2_fields + calc_fields
f_idx = {name: i for i, name in enumerate(update_fields)}

with arcpy.da.UpdateCursor(rect_fc, update_fields) as cursor:
    for row in cursor:
        uid = row[f_idx[custom_uid_field]]
        orig_area = row[f_idx[orig_area_field]] if row[f_idx[orig_area_field]] else 0
        shape_area = row[f_idx["SHAPE@AREA"]]
        
        # A. 範圍標記
        row[f_idx[out_boundary_field]] = 0 if uid in within_uids else 1
        
        # B. T1 & T2 數據
        area_t1 = results_t1.get(uid, {}).get("Total_Area_T1", 0)
        area_t2 = results_t2.get(uid, {}).get("Total_Area_T2", 0)
        
        for f in t1_fields: row[f_idx[f]] = results_t1.get(uid, {}).get(f, 0)
        for f in t2_fields: row[f_idx[f]] = results_t2.get(uid, {}).get(f, 0)
        
        # C. 數學計算 (針對空地的生長分析)
        row[f_idx[t2_orig_gap_field]] = area_t2 - orig_area
        row[f_idx[net_growth_field]] = area_t2 - area_t1 - orig_area
        
        # D. 新增：Crevice 數據計算
        if run_raster_analysis:
            count = crevice_results.get(uid, 0)
            c_area = count * pixel_area
            row[f_idx[crevice_amount_field]] = count
            row[f_idx[crevice_area_field]] = c_area
            # 使用 Shape_Area 為分母
            row[f_idx[crevice_percent_field]] = (c_area / shape_area * 100) if shape_area > 0 else 0
            
        cursor.updateRow(row)

# --- 9. 輸出 CSV ---
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

csv_output_full = os.path.join(output_folder, csv_name)
print(f"正在匯出隨機方格分析 CSV 到: {csv_output_full}")
arcpy.conversion.TableToTable(rect_fc, output_folder, csv_name)

# 清理
arcpy.management.Delete("rect_lyr")
if arcpy.Exists("memory/zonal_crevice"): arcpy.management.Delete("memory/zonal_crevice")

print("🎉 隨機方格分析與 Crevice 監測任務完成！")

🌟 啟用的地圖: Taiwan
📂 當前資料庫 (GDB): D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb
正在根據 Parent_Point_ID 更新 Recruit_UID...
解析到 12 種珊瑚類別。
正在運算 T1 的重疊面積...
正在運算 T2 的重疊面積...
正在執行 Raster Crevice 分析...
📊 Raster 解析度: 0.0010 x 0.0010 m | 單像素面積: 0.000001 m²
檢查範圍邊界...
正在計算並寫入數據...
正在匯出隨機方格分析 CSV 到: D:\ArcGIS\Morpho_Spatial_Demography\output\Random_Squares_Cover_Analysis.csv
🎉 隨機方格分析與 Crevice 監測任務完成！


In [11]:
##在限制範圍內全區打點 沒有排除珊瑚的規則

import arcpy
import os

def generate_random_points_in_boundary():
    # ================= 參數設定區 =================
    boundary_fc = "OL_P1_plot_6m"      # 1. 限制範圍圖層 (隨機點將產生在此範圍內)
    num_points = 200                   # 2. 隨機點數量
    min_dist = "0.1 Meters"            # 3. 點與點之間最小距離
    
    output_folder = arcpy.env.workspace      # 輸出位置 (預設為當前 GDB)
    out_points_name = "Random_Points_Boundary"  # 輸出點圖層名稱
    # =============================================

    arcpy.env.overwriteOutput = True
    
    try:
        # 檢查工作空間是否存在
        if not output_folder:
            print("錯誤: 尚未設定 arcpy.env.workspace。")
            return

        print(f"正在 {boundary_fc} 範圍內產生 {num_points} 個隨機點...")
        
        # 直接使用 boundary_fc 作為限制範圍 (constraining_feature_class)
        arcpy.management.CreateRandomPoints(
            out_path=output_folder, 
            out_name=out_points_name, 
            constraining_feature_class=boundary_fc, 
            number_of_points_or_field=num_points,
            minimum_allowed_distance=min_dist
        )

        full_output_path = os.path.join(str(output_folder), out_points_name)
        print(f"✅ 成功！隨機點已儲存於: {full_output_path}")

    except arcpy.ExecuteError:
        print(arcpy.GetMessages(2))
    except Exception as e:
        print(f"發生錯誤: {e}")

if __name__ == "__main__":
    generate_random_points_in_boundary()

正在 OL_P1_plot_6m 範圍內產生 200 個隨機點...
✅ 成功！隨機點已儲存於: D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\Random_Points_Boundary


In [2]:
##僅排除T1珊瑚圖層產生隨機點
import arcpy
import os

def generate_non_coral_random_points():
    # ================= 自動偵測環境區 =================
    try:
        # 獲取目前開啟的 ArcGIS Pro 專案
        aprx = arcpy.mp.ArcGISProject("CURRENT")
        # 獲取預設的 Geodatabase (GDB) 路徑
        default_gdb = aprx.defaultGeodatabase
        # 獲取目前作用中的地圖名稱
        active_map = aprx.activeMap.name
        
        # 設定 arcpy 工作空間為目前的 GDB
        arcpy.env.workspace = default_gdb
        arcpy.env.overwriteOutput = True
        
        print(f"--- 環境回報 ---")
        print(f"當前地圖: {active_map}")
        print(f"當前 GDB: {default_gdb}")
        print(f"---------------")
        
    except Exception as e:
        print(f"❌ 無法讀取目前地圖環境: {e}")
        return

    # ================= 參數設定區 =================
    # 確保這些圖層已存在於目前的地圖或 GDB 中
    boundary_fc = "OL_P1_plot_6m"           
    coral_t1 = "OL_P1_2204_coral_Simplify"  
    
    num_points = 200                         
    min_dist = "0.1 Meters"                  
    
    out_points_name = "Random_Points_OLP12204_NC"
    # =============================================

    try:
        # 步驟 1: 扣除 T1 珊瑚範圍
        print("步驟 1/2: 正在從邊界中扣除 T1 珊瑚範圍以取得空地...")
        
        # 檢查圖層是否存在
        if not arcpy.Exists(boundary_fc) or not arcpy.Exists(coral_t1):
            print(f"❌ 錯誤: 找不到圖層 {boundary_fc} 或 {coral_t1}，請確認圖層已加入地圖。")
            return

        empty_area = "memory/empty_area"
        arcpy.analysis.Erase(
            in_features=boundary_fc, 
            erase_features=coral_t1, 
            out_feature_class=empty_area
        )

        # 步驟 2: 產生隨機點
        print(f"步驟 2/2: 正在產生 {num_points} 個隨機點...")
        
        arcpy.management.CreateRandomPoints(
            out_path=default_gdb, 
            out_name=out_points_name, 
            constraining_feature_class=empty_area, 
            number_of_points_or_field=num_points,
            minimum_allowed_distance=min_dist
        )

        full_output_path = os.path.join(default_gdb, out_points_name)
        print(f"✅ 成功！隨機點已儲存於: {full_output_path}")
        
        # (選用) 將結果自動加入地圖
        # aprx.activeMap.addDataFromPath(full_output_path)

    except arcpy.ExecuteError:
        print(arcpy.GetMessages(2))
    except Exception as e:
        print(f"發生錯誤: {e}")

if __name__ == "__main__":
    generate_non_coral_random_points()

--- 環境回報 ---
當前地圖: Taiwan
當前 GDB: D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb
---------------
步驟 1/2: 正在從邊界中扣除 T1 珊瑚範圍以取得空地...
步驟 2/2: 正在產生 200 個隨機點...
✅ 成功！隨機點已儲存於: D:\ArcGIS\Morpho_Spatial_Demography\MSD_Taiwan.gdb\Random_Points_OLP12204_NC
